# R5 validation: is the "beyond the panel" test valid and powered? (Phase 0, true data only)

**No models, no predictions.** Every input here is measured data: the R5 test is checked with controls whose answer is known.

**One mask source.** Truth, the cell's own features and its neighbour features are all averaged from the same relabelled mask arrays
(`danenberg-cells-prep`, `jackson-cells-prep`), never from the published cell tables. A hash of every mask is saved for Phase 1.

**Estimator.** For target marker M (one of the 11 shared markers), with the other 13 channels as inputs:
- patients are split three ways: **S** (20%) only fits the control functions; **A / B** (40% / 40%) carry the R5 cross-fit;
- **R5 = ΔR²** between a LightGBM fitted on the conditioning set C plus the candidate prediction and one fitted on C alone,
  fitted on A and scored on B and vice versa, after demeaning both predictions within the target's grouping;
  patient-bootstrap CIs (1000×, over the A ∪ B patients).
- **Why S exists** (found on synthetic data before this run): a control built by cross-fitting is a *different* function of C on
  the rows the model is fitted on and the rows it is scored on, and the full model learns the first function's quirks; that gave
  state-mean R5 ≈ −0.011. A real image model is one fixed function for every cell, so every control that is a function of C is
  fitted once on S and applied to all cells (state −0.003 / +0.000 on synthetic data). The state deciles are defined the same way.

| Rung | Target | Conditioning set C |
|---|---|---|
| R5_own | M − image mean | own 13 channels (raw and image-demeaned) + log pixel count |
| R5_cell | M − image × state-decile mean (state = out-of-fold R5_own fit) | as R5_own, + group-demeaned channels |
| R5_table | as R5_cell | R5_cell set + ring means (non-touching cells ≤ 30 µm) and contact-neighbour means of the 13 channels + counts |

**Controls** (candidate "predictions" built from true data):
- negative: cyclically shifted true map; state mean (the rung's model fitted on S); kNN mean of M over the 30 nearest S cells in
  the own-channel space; truth shuffled within image × state decile; planted α = 0 (state + pure noise);
- positive: identity; planted signal `state + α·residual + √(1−α²)·SD(residual)·noise`, α ∈ {0.05, 0.1, 0.2, 0.4}.

**Noise ceiling.** M averaged over two halves of each cell's pixels (checkerboard (x + y) mod 2, and random uniform). Split-half
reliability of the rung residual, Spearman–Brown corrected, times the residual share = the largest R5 a prediction could reach.

**Spillover check.** All features and the truth recomputed under masks eroded by 1 and 2 px; the neighbour (context) gain
R²(table) − R²(own) is reported per mask.

**Gate G-R5 (pre-registered, all must hold, no partial pass).**
1. Every negative control: |R5 point estimate| < 0.01 on every rung where it is expected to be 0
   (shuffled: R5_cell and R5_table; the others: all three rungs).
2. Identity: R5 ≥ 0.9 × residual share (1 − base R²) on every rung, and ≥ the noise ceiling.
3. Planted signal, for main markers (noise ceiling ≥ 0.05 on R5_cell with both pixel splits), on R5_cell and R5_table:
   point estimates non-decreasing in α (tolerance 0.002) and CI lower bound > 0 at α = 0.1.

---
**Version 2 (2026-10-22): five fixes, decided after version 1 failed G-R5 (116 failures) and before any version-2 result.**
The gate thresholds are unchanged, except fix 5.
1. **Same fit for base and full model.** Both are fitted on the same training subsample with the same seed, so a ΔR² cannot
   come from a different draw of cells or trees.
2. **Cofactor 1, then clip.** Values are arcsinh(x / 1) instead of arcsinh(x / 5), clipped per channel at the 99.9th percentile
   of the S cells. In Jackson, the top 0.1% of cells held 30–43% of the within-image sum of squares for CD3, CK5 and CD20.
3. **The base model knows the group's level of M.** Every rung's conditioning set gets the mean and SD of the true M over the
   rung's group: the image for R5_own; image × state decile for R5_cell and R5_table. These are constant within a group, so they
   carry no cell-level information. Version 1 failed because a candidate can reveal exactly this level (shuffled, shifted).
   *Changed from the v1 summary's proposal:* a leave-one-out group mean would leak the target. Within a group it equals
   const − y_i / (n − 1), an exact copy of the cell's deviation.
4. **The candidate enters only as its deviation from its group mean** (p − group mean(p)), never as raw p.
5. **Identity criterion:** keep "R5 ≥ 0.9 × residual share". Drop "R5 ≥ split-half ceiling". The ceiling counts base-model misfit
   as reliable variance, so it is an upper bound and is reported only.


In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, hashlib, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy import ndimage as ndi
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs"], check=True)   # newer Python images
import tifffile, imagecodecs
print("python", sys.version.split()[0], "| numpy", np.__version__, "| imagecodecs", imagecodecs.__version__)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1" or os.path.exists(f"{IN}/SMOKE")
SEED = 20261020
N_BOOT = 200 if SMOKE else 1000
MIN_CELLS = 200
COF = 1.0                                     # v2: was 5.0
CLIP_Q = 99.9                                 # v2: per-channel upper clip (S cells)
RING_R = 30.0
N_FIT = 20_000 if SMOKE else 100_000          # training cells per patient half
N_STATE = 10                                  # state deciles (R5_cell / R5_table grouping)
K_KNN = 30
N_KNN_REF = 10_000 if SMOKE else 50_000
ALPHAS = [0.0, 0.05, 0.1, 0.2, 0.4]
MASKS = ["full", "erode1", "erode2"]
SMOKE_IMAGES = 60
BUDGET_MIN = 660.0                            # stop optional blocks before Kaggle's 12 h limit
R5_BUDGET_MIN = 560.0                         # no new R5 marker starts after this
LGB = dict(n_estimators=60 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
# Gate thresholds, fixed before the run
NEG_TOL, ID_FRAC, MAIN_CEIL, MONO_TOL = 0.01, 0.9, 0.05, 0.002

NUC = ["histone_h3", "dna1", "dna2"]
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # Danenberg image channel 12 is HER2 clone 3B5
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)
assert len(MARKERS) == 11, MARKERS
IN_SET = NUC + MARKERS                                    # 14 channels, same names in both cohorts
MPOS = [IN_SET.index(m) for m in MARKERS]
RUN_MARKERS = MARKERS[:3] if SMOKE else MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()
FAILED = []

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def minutes():
    return (time.time() - T0) / 60

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

print("SMOKE" if SMOKE else "FULL", "| cpus", os.cpu_count(), "| markers", RUN_MARKERS)

## Cell tables (cell identity, patient, position, contact graph only; no intensities are taken from them)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p))
print(json.dumps(P, indent=1))

tifs = g("*.tiff")
IMGP = {"danenberg": {}, "jackson": {}}
MASKP = {"danenberg": {}, "jackson": {}}
for p in tifs:
    q = p.replace("\\", "/")
    name = os.path.splitext(os.path.basename(p))[0]
    co = "danenberg" if name.startswith("MB_") else "jackson"
    (MASKP if "/masks/" in q else IMGP)[co][name] = p
print({c: (len(IMGP[c]), len(MASKP[c])) for c in IMGP})

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    have = cells.image_name.isin(set(IMGP[key]) & set(MASKP[key]))
    print(f"{key}: cells in images with a tiff and a mask {have.mean():.4f}")
    cells = cells[have]
    n_per = cells.groupby("image_name").size()
    keep = n_per.index[n_per >= MIN_CELLS]
    if SMOKE:
        keep = np.sort(np.random.default_rng(SEED).choice(np.asarray(keep), min(SMOKE_IMAGES, len(keep)), replace=False))
    cells = cells[cells.image_name.isin(keep)]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    patient = cells[pid].astype(str).values
    pats, pcode = np.unique(patient, return_inverse=True)
    u3 = np.random.default_rng(SEED + 3).random(len(pats))
    ev_p = u3 >= 0.2                                     # S = first 20% of patients: control functions only
    W = np.zeros((N_BOOT, len(pats)))
    W[:, ev_p] = np.random.default_rng(SEED + 4).multinomial(int(ev_p.sum()), np.full(int(ev_p.sum()), 1 / ev_p.sum()),
                                                             size=N_BOOT)
    return dict(key=key, cells=cells, img=img_code.astype(np.int64), img_names=np.asarray(img_names), starts=starts,
                patient=patient, pcode=pcode, npat=len(pats), ev=ev_p[pcode], half=(u3 < 0.6)[pcode], W=W,
                split=np.where(~ev_p, "S", np.where(u3 < 0.6, "A", "B"))[pcode],
                xy=cells[["x", "y"]].values.astype(np.float64), ca=ca, cb=cb,
                chan=D_CHAN if key == "danenberg" else J_CHAN)

COH = {k: load_table(k) for k in ("danenberg", "jackson")}
for k, D in COH.items():
    print(f"{k}: {len(D['cells']):,} cells, {len(D['img_names'])} images, {D['npat']} patients, "
          f"cells by split {pd.Series(D['split']).value_counts().to_dict()}, {len(D['ca']):,} contacts")
tick("tables loaded")

## Pixel pass: per-cell means under full / eroded masks, half-pixel means, shifted truth, mask hashes

In [ ]:
def erode_once(mask):
    """Drop every pixel whose 3x3 neighbourhood is not entirely its own label (background counts as a label)."""
    mn = ndi.minimum_filter(mask, size=3, mode="nearest")
    mx = ndi.maximum_filter(mask, size=3, mode="nearest")
    return np.where(mn == mx, mask, 0)

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def means(li, vals, n):
    """Per-cell means of vals[:, j] over pixels with cell index li; NaN where a cell has no pixel."""
    cnt = np.bincount(li, minlength=n)
    out = np.stack([np.bincount(li, vals[:, j], n) for j in range(vals.shape[1])], 1) / np.maximum(cnt, 1)[:, None]
    out[cnt == 0] = np.nan
    return out.astype(np.float32), cnt.astype(np.int32)

def image_pass(img_path, mask_path, obj, n_chan, chans, seed):
    mask_raw = tifffile.imread(mask_path)
    h = hashlib.md5(np.ascontiguousarray(mask_raw).tobytes()).hexdigest()
    mask = mask_raw.astype(np.int64)
    im = read_hwc(img_path, n_chan)[..., chans]                       # H, W, 14 (IN_SET order)
    assert im.shape[:2] == mask.shape, (img_path, im.shape, mask.shape)
    H_, W_ = mask.shape
    n = len(obj)
    lut = np.full(max(int(mask.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(n)
    flat = im.reshape(-1, im.shape[-1])
    out = {"hash": h, "shape": f"{H_}x{W_}"}
    mk = mask
    for v in MASKS:
        if v != "full":
            mk = erode_once(mk)
        loc = lut[mk.ravel()]
        sel = np.flatnonzero(loc >= 0)
        out[f"X_{v}"], out[f"cnt_{v}"] = means(loc[sel], flat[sel], n)
        if v == "full":
            sel_f, li_f = sel, loc[sel]
    yy, xx = np.divmod(sel_f, W_)
    parts = {"checker": ((yy + xx) % 2) == 1,
             "random": np.random.default_rng(seed).random(H_ * W_)[sel_f] < 0.5}
    mvals = flat[sel_f][:, MPOS]
    for name, part in parts.items():
        out[f"half_{name}_A"], _ = means(li_f[part], mvals[part], n)
        out[f"half_{name}_B"], _ = means(li_f[~part], mvals[~part], n)
    rr = np.random.default_rng(seed + 1)
    dy, dx = int(rr.integers(H_ // 4, 3 * H_ // 4 + 1)), int(rr.integers(W_ // 4, 3 * W_ // 4 + 1))
    sh = np.roll(im[..., MPOS], (dy, dx), (0, 1)).reshape(-1, len(MPOS))
    out["tshift"], _ = means(li_f, sh[sel_f], n)
    return out

PIX = {}
for k, D in COH.items():
    chans = [D["chan"].index(c) for c in IN_SET]
    obj = D["cells"].object_number.values.astype(np.int64)
    st = D["starts"]
    res = Parallel(n_jobs=os.cpu_count())(
        delayed(image_pass)(IMGP[k][s], MASKP[k][s], obj[a:b], len(D["chan"]), chans, SEED + 7 * i)
        for i, (s, a, b) in enumerate(zip(D["img_names"], st[:-1], st[1:])))
    R = {key: np.concatenate([r[key] for r in res]) for key in res[0] if key not in ("hash", "shape")}
    pd.DataFrame(dict(cohort=k, image_name=D["img_names"], mask_md5=[r["hash"] for r in res],
                      shape=[r["shape"] for r in res])).to_csv(f"{OUT}/mask_hashes_{k}.csv", index=False)
    del res
    for v in MASKS:
        print(f"{k} {v}: cells with ≥1 pixel {np.mean(R[f'cnt_{v}'] > 0):.4f}, "
              f"median pixels {np.median(R[f'cnt_{v}']):.0f}")
    assert np.mean(R["cnt_full"] > 0) > 0.95, "mask relabelling does not match the cell table"
    chk = []
    for c in ("histone_h3", "dna1", "pan_cytokeratin", "estrogen_receptor_alpha"):
        tc = D_TABLE.get(c, c) if k == "danenberg" else c
        a, b = R["X_full"][:, IN_SET.index(c)], D["cells"][tc].values.astype(np.float64)
        ok = np.isfinite(a) & np.isfinite(b)
        chk.append(dict(cohort=k, channel=c, r=float(np.corrcoef(a[ok], b[ok])[0, 1])))
    chk = pd.DataFrame(chk)
    print(chk.round(3).to_string(index=False))
    assert chk.r.median() > 0.6, "image-derived cell means disagree with the published table (channel order or masks)"
    PIX[k] = R
    tick(f"pixel pass {k}")

for k, D in COH.items():
    R = PIX[k]
    cols = {"image_name": D["cells"].image_name.values, "object_number": D["cells"].object_number.values,
            "patient": D["patient"], "r5_split": D["split"], "n_pixels": R["cnt_full"]}
    cols.update({f"raw__{c}": R["X_full"][:, j] for j, c in enumerate(IN_SET)})
    pd.DataFrame(cols).to_parquet(f"{OUT}/percell_full_mask_{k}.parquet", index=False)
tick("per-cell tables saved")

## Features, neighbours and the cross-fit machinery

In [ ]:
def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

def gmean(v, key):
    """Group mean of v (1-D or 2-D) broadcast back to rows; NaNs ignored."""
    k = int(key.max()) + 1
    if v.ndim == 1:
        ok = np.isfinite(v)
        s = np.bincount(key[ok], v[ok], k)
        c = np.bincount(key[ok], minlength=k)
        return (s / np.maximum(c, 1))[key].astype(np.float32)
    return np.column_stack([gmean(v[:, j], key) for j in range(v.shape[1])])

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_graph(D, C):
    """Per image: non-touching cells within RING_R of the centroid (block-diagonal sparse matrix)."""
    blocks = []
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        R = (R - R.multiply(C[a:b, a:b])).tocsr()
        R.eliminate_zeros()
        blocks.append(R)
    return sp.block_diag(blocks, format="csr")

def nbr_mean(G, X):
    d = np.asarray(G.sum(1)).ravel()
    Xz = np.nan_to_num(X)
    w = G @ np.isfinite(X).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        M = (G @ Xz) / w
    M[w == 0] = np.nan
    return M.astype(np.float32), d.astype(np.float32)

GRAPH = {}
for k, D in COH.items():
    C = contact_matrix(D)
    GRAPH[k] = dict(C=C, R=ring_graph(D, C))
    print(f"{k}: mean contacts {GRAPH[k]['C'].sum(1).mean():.2f}, mean ring cells {GRAPH[k]['R'].sum(1).mean():.2f}")
tick("graphs")

def oof(F, y, D, seed):
    """Cross-fit on the evaluation patients: fitted on A, predicted on B and vice versa (NaN on S)."""
    out = np.full(len(y), np.nan, np.float32)
    rng = np.random.default_rng(seed)
    for h in (False, True):
        tr = np.flatnonzero(D["ev"] & (D["half"] == h) & np.isfinite(y))
        tr = np.sort(rng.choice(tr, min(N_FIT, len(tr)), replace=False))
        te = np.flatnonzero(D["ev"] & (D["half"] != h))
        mdl = lgb.LGBMRegressor(random_state=int(seed) % (2 ** 31), **LGB).fit(F[tr], y[tr])
        out[te] = mdl.predict(F[te])
    return out

def fit_fixed(F, y, D, seed):
    """One fixed function of F for every cell: fitted on the S patients only."""
    rng = np.random.default_rng(seed)
    tr = np.flatnonzero(~D["ev"] & np.isfinite(y))
    tr = np.sort(rng.choice(tr, min(N_FIT, len(tr)), replace=False))
    mdl = lgb.LGBMRegressor(random_state=int(seed) % (2 ** 31), **LGB).fit(F[tr], y[tr])
    return mdl.predict(F).astype(np.float32)

def sse_parts(target, pred, key, pcode, npat):
    """Per-patient SSE after demeaning the prediction within the target's grouping, and per-patient SST."""
    p = pred - gmean(pred, key)
    ok = np.isfinite(target) & np.isfinite(p)
    return (np.bincount(pcode[ok], (target[ok] - p[ok]) ** 2, npat),
            np.bincount(pcode[ok], target[ok] ** 2, npat))

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)

def knn_mean(Z, y, D, seed):
    """Mean y of the K_KNN nearest S cells in Z space (a nonparametric state, one fixed function for every cell)."""
    rng = np.random.default_rng(seed)
    Zc = np.nan_to_num(Z)
    ref = np.flatnonzero(~D["ev"] & np.isfinite(y))
    ref = np.sort(rng.choice(ref, min(N_KNN_REF, len(ref)), replace=False))
    _, nn = cKDTree(Zc[ref]).query(Zc, k=K_KNN, workers=-1)
    return y[ref][nn].mean(1).astype(np.float32)

def shuffle_within(v, key, seed):
    r = np.random.default_rng(seed).random(len(v))
    s1 = np.lexsort((np.arange(len(v)), key))
    s2 = np.lexsort((r, key))
    out = np.empty_like(v)
    out[s1] = v[s2]
    return out

def arc(X):
    return np.arcsinh(np.asarray(X, np.float32) / COF).astype(np.float32)

CLIP = {}

def clip_hi(k, v):
    """v2: per-channel upper clip, the CLIP_Q percentile of the S cells (one fixed transform; no evaluation cell sets it)."""
    if (k, v) not in CLIP:
        A = arc(PIX[k][f"X_{v}"])
        CLIP[(k, v)] = np.nanpercentile(A[~COH[k]["ev"]], CLIP_Q, axis=0).astype(np.float32)
    return CLIP[(k, v)]

def gstats(y, key):
    """v2: mean and SD of y over each group, broadcast to rows (constant within a group)."""
    m = gmean(y, key)
    return np.c_[m, np.sqrt(np.maximum(gmean(y * y, key) - m * m, 0))].astype(np.float32)

def build(k, v, mi):
    """Targets, groupings and conditioning sets for cohort k, mask variant v, marker index mi."""
    D, R, G = COH[k], PIX[k], GRAPH[k]
    A = np.minimum(arc(R[f"X_{v}"]), clip_hi(k, v))
    kk = IN_SET.index(MARKERS[mi])
    cols = [j for j in range(len(IN_SET)) if j != kk]
    img = D["img"]
    y = A[:, kk].copy()
    logc = np.log(np.maximum(R[f"cnt_{v}"], 1)).astype(np.float32)[:, None]
    Ci = A[:, cols]
    own = np.c_[Ci, Ci - gmean(Ci, img), logc].astype(np.float32)
    ring, nring = nbr_mean(G["R"], Ci)
    cont, ncont = nbr_mean(G["C"], Ci)
    nbr = np.c_[ring, cont, nring[:, None], ncont[:, None]].astype(np.float32)
    return dict(y=y, img=img, own=own, nbr=nbr, Z=harm(np.c_[Ci, logc]))

def state_groups(own, y, img, D, seed):
    """Within-image target t, image x state-decile key (state = fixed S model of t on own channels) and target u."""
    t = (y - gmean(y, img)).astype(np.float32)
    s = fit_fixed(own, t, D, seed)
    dec = pd.qcut(pd.Series(s).rank(method="first"), N_STATE, labels=False).values.astype(np.int64)
    gkey = img * N_STATE + dec
    u = (y - gmean(y, gkey)).astype(np.float32)
    return t, gkey, u

def score(tgt, pred_b, pred_f, key, D):
    """ΔR² (point, bootstrap vector) and base R² on the evaluation patients."""
    te = np.where(D["ev"], tgt, np.nan)
    sse_b, sst = sse_parts(te, pred_b, key, D["pcode"], D["npat"])
    sse_f, _ = sse_parts(te, pred_f, key, D["pcode"], D["npat"])
    return ((sse_b.sum() - sse_f.sum()) / sst.sum(), (D["W"] @ sse_b - D["W"] @ sse_f) / (D["W"] @ sst),
            1 - sse_b.sum() / sst.sum())

def r5_marker(D, own, nbr, y, img, Z, tsh, halves, seed, rungs=("R5_own", "R5_cell", "R5_table"), extra=None):
    """All R5 rungs, controls, planted signal and noise ceiling for one marker. Returns (rows, ceiling rows)."""
    t, gkey, u = state_groups(own, y, img, D, seed)
    n_c = own.shape[1] // 2
    Ci = own[:, :n_c]
    own_g = np.c_[own, Ci - gmean(Ci, gkey)].astype(np.float32)
    gs_img, gs_grp = gstats(y, img), gstats(y, gkey)              # v2 fix 3: the group's level of M, for base and full
    RUNG = {"R5_own": (np.c_[own, gs_img].astype(np.float32), t, img),
            "R5_cell": (np.c_[own_g, gs_grp].astype(np.float32), u, gkey),
            "R5_table": (np.c_[own_g, nbr, gs_grp].astype(np.float32), u, gkey)}
    knn = knn_mean(Z, y, D, seed + 3)
    shuf = shuffle_within(y, gkey, seed + 4)
    noise = np.random.default_rng(seed + 5).standard_normal(len(y)).astype(np.float32)
    rows, ceil = [], []
    for ri, rung in enumerate(rungs):
        F, tgt, key = RUNG[rung]
        pb = oof(F, tgt, D, seed + 10 + ri)
        sfix = fit_fixed(F, tgt, D, seed + 20 + ri)
        r = (tgt - (pb - gmean(pb, key))).astype(np.float32)
        sd_r = float(np.nanstd(r[D["ev"]]))
        _, _, base_r2 = score(tgt, pb, pb, key, D)
        for sp_, (yA, yB) in halves.items():
            rA = (yA - gmean(yA, key)) - (pb - gmean(pb, key))
            rB = (yB - gmean(yB, key)) - (pb - gmean(pb, key))
            ok = np.isfinite(rA) & np.isfinite(rB) & D["ev"]
            rho = float(np.sum(rA[ok] * rB[ok]) / np.sqrt(np.sum(rA[ok] ** 2) * np.sum(rB[ok] ** 2)))
            rel = 2 * rho / (1 + rho) if rho > -1 else np.nan
            ceil.append(dict(rung=rung, split=sp_, rho_half=rho, reliability=rel, resid_share=1 - base_r2,
                             ceiling=max(rel, 0) * (1 - base_r2), n_cells=int(ok.sum())))
        preds = {"shifted true map": tsh, "state mean": sfix, "kNN mean (S cells)": knn,
                 "shuffled within image x state": shuf, "identity": y}
        for a in ALPHAS:
            preds[f"planted a={a:g}"] = sfix + a * r + np.sqrt(1 - a * a) * sd_r * noise
        if extra:
            preds.update(extra)
        for pi, (name, p) in enumerate(preds.items()):
            if p is None:
                continue
            p = np.asarray(p, np.float32)
            pf = oof(np.c_[F, p - gmean(p, key)], tgt, D, seed + 10 + ri)   # v2 fixes 4 and 1
            d, boot, _ = score(tgt, pb, pf, key, D)
            lo, hi = ci(boot)
            rows.append(dict(rung=rung, control=name, dR2=d, lo=lo, hi=hi, base_r2=base_r2,
                             resid_share=1 - base_r2, dR2_frac=d / max(1 - base_r2, 1e-9)))
    return rows, ceil
tick("helpers ready")

## R5 on true data: controls, planted signal, noise ceiling (full masks)

In [ ]:
ROWS, CEIL = [], []
for k, D in COH.items():
    for mi, m in enumerate(MARKERS):
        if m not in RUN_MARKERS:
            continue
        if minutes() > R5_BUDGET_MIN:
            print(f"R5 budget reached; skipping {k} {m}")
            FAILED.append(f"r5 skipped (budget) {k} {m}")
            continue
        try:
            seed = SEED + 1000 * (k == "jackson") + 10 * mi
            B = build(k, "full", mi)
            hi = clip_hi(k, "full")[IN_SET.index(m)]
            tsh = np.minimum(arc(PIX[k]["tshift"][:, mi]), hi)
            halves = {sp_: (np.minimum(arc(PIX[k][f"half_{sp_}_A"][:, mi]), hi),
                            np.minimum(arc(PIX[k][f"half_{sp_}_B"][:, mi]), hi))
                      for sp_ in ("checker", "random")}
            rows, ceil = r5_marker(D, B["own"], B["nbr"], B["y"], B["img"], B["Z"], tsh, halves, seed)
            ROWS += [dict(cohort=k, marker=m, **r) for r in rows]
            CEIL += [dict(cohort=k, marker=m, **c) for c in ceil]
            pd.DataFrame(ROWS).to_csv(f"{OUT}/r5_controls.csv", index=False)       # checkpoint after every marker
            pd.DataFrame(CEIL).to_csv(f"{OUT}/noise_ceiling.csv", index=False)
            tick(f"{k} {m}: done")
        except Exception:
            print(f"!!! {k} {m} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(f"r5 {k} {m}")
        gc.collect()

RES = pd.DataFrame(ROWS)
CEILD = pd.DataFrame(CEIL)
RES.to_csv(f"{OUT}/r5_controls.csv", index=False)
CEILD.to_csv(f"{OUT}/noise_ceiling.csv", index=False)
with pd.option_context("display.width", 250, "display.max_rows", 500):
    print(RES.pivot_table(index=["cohort", "marker", "control"], columns="rung", values="dR2").round(4).to_string())
    print(CEILD.pivot_table(index=["cohort", "marker", "rung"], columns="split", values="ceiling").round(4).to_string())
tick("R5 controls done")

## Spillover check: neighbour (context) gain under full and eroded masks

In [ ]:
SPILL = []
for k, D in COH.items():
    for mi, m in enumerate(MARKERS):
        if m not in RUN_MARKERS:
            continue
        for v in MASKS:
            if minutes() > BUDGET_MIN:
                print(f"budget reached; spillover check stops at {k} {m} {v}")
                break
            try:
                seed = SEED + 5000 + 1000 * (k == "jackson") + 10 * mi + MASKS.index(v)
                B = build(k, v, mi)
                y, img = B["y"], B["img"]
                t, gkey, u = state_groups(B["own"], y, img, D, seed)
                Ci = B["own"][:, :len(IN_SET) - 1]
                own_g = np.c_[B["own"], Ci - gmean(Ci, gkey)].astype(np.float32)
                tab_g = np.c_[own_g, B["nbr"]].astype(np.float32)
                row = dict(cohort=k, marker=m, mask=v, n_cells=int(np.isfinite(y).sum()))
                for j, (nm, (F, tgt, key)) in enumerate({"own_t": (B["own"], t, img), "own_u": (own_g, u, gkey),
                                                         "table_u": (tab_g, u, gkey),
                                                         "table_t": (np.c_[B["own"], B["nbr"]], t, img)}.items()):
                    pb = oof(F, tgt, D, seed + 17)
                    row[f"r2_{nm}"] = score(tgt, pb, pb, key, D)[2]
                row["context_gain_t"] = row["r2_table_t"] - row["r2_own_t"]
                row["context_gain_u"] = row["r2_table_u"] - row["r2_own_u"]
                SPILL.append(row)
            except Exception:
                print(f"!!! spillover {k} {m} {v} failed:\n{traceback.format_exc()}", flush=True)
                FAILED.append(f"spill {k} {m} {v}")
        tick(f"spillover {k} {m}")
SPILLD = pd.DataFrame(SPILL)
SPILLD.to_csv(f"{OUT}/spillover_by_mask.csv", index=False)
if len(SPILLD):
    print(SPILLD.pivot_table(index=["cohort", "marker"], columns="mask", values="context_gain_u").round(4).to_string())

## Gate G-R5

In [ ]:
NEG_ALL = ["shifted true map", "state mean", "kNN mean (S cells)", "planted a=0"]
NEG_GROUPED = ["shuffled within image x state"]
fails = []
neg = RES[(RES.control.isin(NEG_ALL)) | (RES.control.isin(NEG_GROUPED) & RES.rung.isin(["R5_cell", "R5_table"]))]
bad = neg[neg.dR2.abs() >= NEG_TOL]
for _, r in bad.iterrows():
    fails.append(f"negative control {r.control} {r.cohort} {r.marker} {r.rung}: R5 {r.dR2:+.4f}")

cmin = CEILD.pivot_table(index=["cohort", "marker", "rung"], columns="split", values="ceiling")
cmax_ = cmin.max(1)
cdiff = (cmin.max(1) - cmin.min(1))
flag_split = cdiff[cdiff > 0.05]
cmin = cmin.min(1)
idt = RES[RES.control == "identity"].set_index(["cohort", "marker", "rung"])
ID_BELOW_CEIL = []
for ix, r in idt.iterrows():
    if r.dR2 < ID_FRAC * r.resid_share:
        fails.append(f"identity {ix}: R5 {r.dR2:.4f} < {ID_FRAC} x residual share {r.resid_share:.4f}")
    if ix in cmax_.index and r.dR2 < cmax_[ix]:          # v2 fix 5: reported only (the ceiling is an upper bound)
        ID_BELOW_CEIL.append(f"{ix}: R5 {r.dR2:.4f} vs ceiling {cmax_[ix]:.4f}")

main = sorted({(c, m) for (c, m, rg), v in cmin.items() if rg == "R5_cell" and v >= MAIN_CEIL})
PLANT = []
for c, m in main:
    for rung in ("R5_cell", "R5_table"):
        sub = RES[(RES.cohort == c) & (RES.marker == m) & (RES.rung == rung) & RES.control.str.startswith("planted")]
        sub = sub.assign(alpha=sub.control.str.replace("planted a=", "").astype(float)).sort_values("alpha")
        mono = bool(np.all(np.diff(sub.dR2.values) >= -MONO_TOL))
        a01 = sub[sub.alpha == 0.1]
        det = bool(len(a01) and a01.lo.iloc[0] > 0)
        PLANT.append(dict(cohort=c, marker=m, rung=rung, monotone=mono, detects_a01=det,
                          dR2_a01=float(a01.dR2.iloc[0]) if len(a01) else np.nan))
        if not mono:
            fails.append(f"planted curve not monotone: {c} {m} {rung}")
        if not det:
            fails.append(f"planted alpha 0.1 not detected: {c} {m} {rung}")
PLANTD = pd.DataFrame(PLANT)
PLANTD.to_csv(f"{OUT}/planted_detection.csv", index=False)
if not main:
    fails.append("no main markers (noise ceiling >= 0.05 on R5_cell): the planted-signal criterion cannot be met")

GATE = dict(passed=(len(fails) == 0 and not FAILED), n_failures=len(fails), failures=fails, failed_blocks=FAILED,
            main_markers=[f"{c}:{m}" for c, m in main],
            ceiling_split_flags=[f"{c}:{m}:{rg} ({d:.3f})" for (c, m, rg), d in flag_split.items()],
            thresholds=dict(NEG_TOL=NEG_TOL, ID_FRAC=ID_FRAC, MAIN_CEIL=MAIN_CEIL, MONO_TOL=MONO_TOL),
            identity_below_ceiling_reported=ID_BELOW_CEIL, version=2, smoke=SMOKE, minutes=minutes())
json.dump(GATE, open(f"{OUT}/gate.json", "w"), indent=1)
print(json.dumps({k: v for k, v in GATE.items() if k != "failures"}, indent=1))
print("\n".join(fails[:60]))
print("GATE G-R5:", "PASS" if GATE["passed"] else "FAIL")

## Figure: control R5 and the planted-signal curve

In [ ]:
try:
    COL = {"danenberg": ("#a1d99b", "#238b45"), "jackson": ("#9ecae1", "#2171b5")}
    rungs = ["R5_own", "R5_cell", "R5_table"]
    fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), sharey="row")
    order = ["shifted true map", "state mean", "kNN mean (S cells)", "shuffled within image x state", "planted a=0"]
    for j, rung in enumerate(rungs):
        ax = axes[0, j]
        for ci_, k in enumerate(("danenberg", "jackson")):
            s = RES[(RES.cohort == k) & (RES.rung == rung) & RES.control.isin(order)]
            agg = s.groupby("control").dR2.agg(["mean", "std"]).reindex(order)
            x = np.arange(len(order)) + (ci_ - 0.5) * 0.38
            ax.bar(x, agg["mean"], 0.36, yerr=agg["std"], color=COL[k][1], label=k.capitalize(), capsize=2)
        ax.axhspan(-NEG_TOL, NEG_TOL, color="0.85", zorder=0)
        ax.set_xticks(np.arange(len(order)))
        ax.set_xticklabels(["Shifted map", "State mean", "kNN mean", "Shuffled\nin state", "Noise"], fontsize=8)
        ax.set_ylabel(f"{rung}: ΔR², mean over markers (SD)")
        ax.tick_params(labelleft=True)
        ax = axes[1, j]
        for k in ("danenberg", "jackson"):
            s = RES[(RES.cohort == k) & (RES.rung == rung) & RES.control.str.startswith("planted")]
            s = s.assign(alpha=s.control.str.replace("planted a=", "").astype(float))
            for m_, sm in s.groupby("marker"):
                sm = sm.sort_values("alpha")
                ax.plot(sm.alpha, sm.dR2_frac, color=COL[k][1], alpha=0.6, lw=1)
        ax.set_xlabel("Planted α (correlation with the true residual)")
        ax.set_ylabel(f"{rung}: ΔR² / residual share")
        ax.tick_params(labelleft=True)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=2, frameon=False)
    for ax in axes.ravel():
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(f"{OUT}/fig_r5_validation.png", dpi=150)
    plt.close(fig)
except Exception:
    print("!!! figure failed:\n" + traceback.format_exc())
    FAILED.append("figure")
json.dump(dict(minutes=minutes(), failed=FAILED, smoke=SMOKE), open(f"{OUT}/run_info.json", "w"), indent=1)
tick("done")